In [6]:
%load_ext dotenv
%dotenv

In [1]:
import pandas as pd
import numpy as np
import pprint
from sentence_transformers import SentenceTransformer
import os
from pinecone import Pinecone, ServerlessSpec

In [10]:
df = pd.read_csv("data/alldata_1_for_kaggle.csv", encoding='latin1')

# Fixing oddly named columns of '0' and 'a'
df = df.rename(columns={"0": "diagnosis", "a": "clinical_text"})

# Drop empty rows to prevent embedding errors
df = df.dropna()

# For testing purposes, let's just use the first 500 rows so it doesn't take hours to embed!
# df = df.head(500).copy()

files["metadata"] = files.apply(lambda row: {
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"],
}, axis = 1)

In [11]:
# Model fine-tuned on PubMed literature
model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
# Specific weights for different text components
weight_diagnosis = 5
weight_abstract = 3
weight_full_text = 1 

# Pinecone metadata to return with search results
df['metadata'] = df.apply(lambda row: {
    "diagnosis": row['diagnosis'],
    # We save a 300-character snippet to display in the search results
    "text_snippet": str(row['clinical_text'])[:300] + "..." 
}, axis=1)

def create_medical_embeddings(row):
    text = str(row['clinical_text'])
    
    # 1. Diagnosis embedding
    emb_diagnosis = model.encode(row['diagnosis'], show_progress_bar=False) * weight_diagnosis
    
    # 2. "Abstract" embedding (First 200 chars usually contain the core thesis)
    abstract = text[:200]
    emb_abstract = model.encode(abstract, show_progress_bar=False) * weight_abstract
    
    # 3. Full Text embedding (1000 chars cap to avoid model token limits)
    full_text = text[:1000] 
    emb_full_text = model.encode(full_text, show_progress_bar=False) * weight_full_text

    # Combine by averaging them based on the weights
    total_weight = weight_diagnosis + weight_abstract + weight_full_text
    combined_embedding = (emb_diagnosis + emb_abstract + emb_full_text) / total_weight
    
    return combined_embedding

print("Generating weighted embeddings takes a few minutes")
df['embedding'] = df.apply(create_medical_embeddings, axis=1)
df['unique_id'] = df.index.astype(str)
print("Embeddings generated!")

Generating weighted embeddings takes a few minutes
Embeddings generated!


In [13]:
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

index_name = "medical-weighted-search"

# IMPORTANT: PubMedBERT outputs 768-dimensional vectors, not 384 like MiniLM
dimension = 768 
metric = "cosine"

# Delete index if it already exists to start fresh
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} successfully deleted.")

# Create the new index
pc.create_index(
    name=index_name, 
    dimension=dimension, 
    metric=metric, 
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

index = pc.Index(index_name)

# Prepare vectors and upsert
vectors_to_upsert = [(row['unique_id'], row['embedding'].tolist(), row['metadata']) for _, row in df.iterrows()]

index.upsert(vectors=vectors_to_upsert)
print("Data successfully upserted to Pinecone index!")

medical-weighted-search successfully deleted.


PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Sun, 01 Mar 2026 05:22:32 GMT', 'Content-Type': 'application/json', 'Content-Length': '127', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '3557', 'x-envoy-upstream-service-time': '4', 'x-pinecone-response-duration-ms': '3560', 'server': 'envoy'})
HTTP response body: {"code":11,"message":"Error, decoded message length too large: found 31854334 bytes, the limit is: 4194304 bytes","details":[]}


In [8]:
# Test with a highly specific medical query
query = "patient presenting with pulmonary nodules and respiratory distress"
query_embedding = model.encode(query, show_progress_bar=False).tolist()

query_results = index.query(
    vector=[query_embedding],
    top_k=5,
    include_metadata=True
)

score_threshold = 0.3

print(f"\n--- SEARCH RESULTS FOR: '{query}' ---\n")

for match in query_results['matches']:
    if match['score'] >= score_threshold:
        details = match.get('metadata', {})
        diagnosis = details.get('diagnosis', 'N/A')
        snippet = details.get('text_snippet', 'No snippet available')
        
        print(f"Score: {match['score']:.4f} | Diagnosis: {diagnosis}")
        print(f"Snippet: {snippet}")
        print("-" * 80)


--- SEARCH RESULTS FOR: 'patient presenting with pulmonary nodules and respiratory distress' ---

Score: 0.4220 | Diagnosis: Thyroid_Cancer
Snippet: pulmonary disease COPD is due to structural changes and narrowing of small airways and parenchymaldestruction loss of the alveolar attachment as a result of pulmonary emphysema which all lead to airï¬ow limitation Extracorporeal shock waves ESW increase cell proliferation and diï¬erentiation of con...
--------------------------------------------------------------------------------
Score: 0.3802 | Diagnosis: Thyroid_Cancer
Snippet: Ofï¬cial Case Reports Journal of the Asian Paciï¬c Society of RespirologyRespirology Case ReportsEndobronchial metastases from a primary embryonalcarcinomaChiKang Teng1ChihYen Tu121Division of Pulmonary and Critical Care Medicine Department of Internal Medicine China Medical University Hospital Ta...
--------------------------------------------------------------------------------
Score: 0.3288 | Diagnosis: Th